# Pré-processamento do mosaico Sentinel-2 (estágio 05)

Constrói, para cada ano requerido (os `year` próprios das fontes habilitadas), o composite multiespectral alinhado e normalizado a partir do mosaico livre de nuvens exportado no estágio 01, alinhando-o ao grid da máscara de referência daquele ano (a primeira fonte habilitada com o mesmo ano) — garantindo sobreposição pixel a pixel para a geração de patches por fonte.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt` (incluindo rasterio, usado na leitura e na escrita dos GeoTIFFs), garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`.

In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src import io
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
print(f"Plataforma: {platform}")
print(f"Composite (alinhado e normalizado): {storage_paths['data_processed_composites']}")
print(f"Figuras: {storage_paths['artifacts_figures']}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões dos pacotes principais do ambiente, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes, flags determinísticas e registra as versões do ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Anos de composite necessários

Lista os anos de dados requeridos pelo pipeline (os `year` próprios das fontes, ex.: MapBiomas 2025, AlphaEarth 2024, Emater 2018); um composite alinhado e normalizado é gerado para cada ano.

In [ ]:
# Lista os anos de dados necessários para a geração dos composites.
from src.data.mask_utils import required_data_years

years = required_data_years()
print(f"Anos de composite requeridos: {', '.join(str(y) for y in years)}")

## Dependências dos estágios 01 e 02

Verifica que o mosaico Sentinel-2 de cada ano (estágio 01) e a máscara de referência de cada ano (estágio 02) estão disponíveis no caminho canônico — sem estas dependências o composite não pode ser alinhado.

In [ ]:
# Verifica as dependências dos estágios 01 (mosaicos por ano) e 02 (máscaras de referência).
from src.data.gee_client import year_dates
from src.data.mask_utils import reference_mask_for_year
from src.data.preprocessing import mosaic_path
from src.utils import check_dependencies

dependencies = {}
for year in years:
    start, end = year_dates(year)
    dependencies[f"Mosaico Sentinel-2 {year} (estágio 01)"] = mosaic_path(storage_paths, start, end)
    dependencies[f"Máscara de referência {year} (estágio 02)"] = reference_mask_for_year(storage_paths, year)
check_dependencies(dependencies)


## Composite alinhado e normalizado por ano

Garante (idempotente) o composite multiespectral de cada ano em `MyDrive/tcc/data/processed/composites/`: o mosaico do estágio 01 é alinhado ao grid da máscara de referência do ano e normalizado para reflectância em [0, 1].

In [ ]:
# Garante o composite alinhado e normalizado de cada ano (reutiliza se já existir).
from src.data.preprocessing import build_preprocessed_composite

composite_files = {
    year: build_preprocessed_composite(storage_paths, year) for year in years
}
composite_file = composite_files[years[-1]]

## Verificação dos composites

Confere o grid, o CRS, o número de bandas, o alinhamento à máscara de referência do ano e o intervalo de reflectância de cada composite.

In [ ]:
# Verifica o composite de cada ano (grid, CRS, bandas, alinhamento e intervalo).
from src.data.preprocessing import verify_composite

composite_stats = {}
for year in years:
    composite_stats[year] = verify_composite(storage_paths, year)
    stats = composite_stats[year]
    vmin, vmax = stats["value_range"]
    print(f"{year}: {stats['shape'][0]}x{stats['shape'][1]} px | bandas {stats['bands']} | "
          f"alinhado {stats['aligned_to_mask']} | reflectância [{vmin:.3f}, {vmax:.3f}]")


## Figura dos composites

Renderiza e persiste a miniatura RGB de cada composite (bandas B4, B3, B2, com realce por percentil) em `MyDrive/tcc/artifacts/figures/`; execuções repetidas reutilizam as figuras existentes.

In [ ]:
# Renderiza e persiste a figura de cada composite (reutiliza se já existir).
from src.data.preprocessing import save_composite_preview

figure_paths = []
for year in years:
    figure_paths.append(save_composite_preview(storage_paths, year))


## Resumo da etapa

Exibe o resumo do pré-processamento: dependências reutilizadas, composites persistidos, métricas de verificação e figuras de registro.

In [ ]:
# Exibe o resumo da etapa de pré-processamento dos composites.
active_stats = composite_stats[years[-1]]
summary = {
    "Anos de composite": ", ".join(str(y) for y in years),
    "Composite ativo": str(composite_file),
    "Bandas": active_stats["bands"],
    "Grid": f"{active_stats['shape'][0]}x{active_stats['shape'][1]} px",
    "Alinhado à máscara de referência do ano": active_stats["aligned_to_mask"],
    "Figuras": ", ".join(str(p) for p in figure_paths),
}
from src.utils import print_summary

print_summary(summary, "05")
